## Ensemble_Agent Experimenting

In [73]:
from deal_hunter.agents.items import Item

In [1]:
#imports
import os 
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.notebook import tqdm 

/Volumes/256 SSD/deal_hunter/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Dataset

In [2]:
#hf login
load_dotenv(override = True)
hf_token = os.environ["HF_TOKEN"]
login(token=hf_token , add_to_git_credential = False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
from pathlib import Path
import sys

root_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0,str(root_dir/"src"))



In [4]:
pricer_data = "Vishy08/pricer-data"
train , test, val = Item.from_hub(dataset_name=pricer_data)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 320,000 training items, 4,000 validation items, 8,000 test items


In [5]:
test[0]

Item(title='ZLINE 30 in. Wooden Wall Mount Range Hood in Cottage White - Includes Motor (KPTT-30)', price=899.95, category='Appliances', test_prompt="How much does this cost to the nearest dollar?\n\nZLINE 30 in. Wooden Wall Mount Range Hood in Cottage White - Includes Motor\nThe ZLINE KPTT wall mount wooden range hood designed to be both elegant and powerful, featuring the industry's only lifetime warranty motor. The hand-finished painted cottage white wood is made from solid pine with a stainless steel inner frame, making it both durable and long-lasting. This durable construction, along with ZLINE's lifetime warranty on the motor, guarantees a range hood that will last for a lifetime. This classic wooden hood contains many modern features, such as Hand-carved, hand-finished wood; Dishwasher-safe stainless steel baffle filters; Built-in LED lighting; High performance motor with speeds up to 400 CFM. All ZLINE range hoods come equipped with everything needed to easily install and use,

In [74]:
import chromadb
DB = "products_vectorstore"
chroma_client = chromadb.PersistentClient(path = DB)

In [75]:
#Using SentenceTransformer Encoding LLM ( Free(fast) + Private)
from sentence_transformers import SentenceTransformer
encoder_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6404.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [76]:
sentences = [
    item.test_prompt for item in test[:50]
]
embeddings = encoder_model.encode(sentences=sentences)


In [77]:
type(embeddings)

numpy.ndarray

In [78]:
def summarizer(item):
    question = "How much does this cost to the nearest dollar?\n\n"
    prefix = "\n\nPrice is $"
    summary = item.test_prompt.replace(question,"").replace(prefix,"")
    return f"{summary}"

In [79]:
summarizer(test[42])

"Moving Out for Xbox One - Xbox One\nBecome a certified removals master in this action, puzzle, physics based moving Simulator that brings new meaning to couch co-op. In moving out players can train alone or with friends to learn the dos and don'ts of moving furniture. Expect plot twists, irreverent humor, cameos and borderline trademark infringing scenarios. You thought moving was dull, think again! Grab your friends and become a certified moving master in moving out! Moving out is currently in development as the premiere title from smog's melbourne studio and is made in collaboration with stockholm-based developer deem games. Co-op! Enjoy the game solo as an independent contractor or team up with some friends for the full story mode. Up to four players can cozy up on the couch together and argue over the best way to move.. A couch! Action! Players will learn the ropes in a"

In [72]:
def build_product_collection(client, encoder_model, items,name = "products",batch_size = 500):

    collection = client.get_or_create_collection(name)